# SN-02 — Perimetre A/B/C

Les personnes morales non validees en phase 1, reparties en trois
sous-ensembles :

- **A** : rattachees a au moins un etablissement ;
- **B** : sans etablissement mais avec un SIREN renseigne ;
- **C** : sans etablissement et sans SIREN.

Le rattachement se lit sur `NB_PmSmsseId` de l'etablissement, un identifiant
entier, la ou l'ancien modele comparait des numeros FINESS sous forme de
chaines — source de faux negatifs de formatage.

L'exclusion porte sur l'identifiant de la PM et non sur le SIREN : deux PM
jumelles partageant un SIREN validee ne sont pas ecartees si elles-memes ne
l'ont pas ete.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.sirenisation import construire_perimetre_abc
from src.display import afficher_tableau
from config.settings import (
    FINESS_PM_CLEAN, FINESS_EGE_CLEAN, SN_VALIDES_P1, SN_PERIMETRE, PROCESSED_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

valides_p1 = pd.read_parquet(SN_VALIDES_P1)
ids_valides = set(valides_p1["NB_PmSmsseId"].astype(str))
print(f"PM validees en phase 1 : {len(ids_valides):,}")

PM validees en phase 1 : 38,785


In [2]:
df_pm = pd.read_parquet(FINESS_PM_CLEAN)
df_ege = pd.read_parquet(FINESS_EGE_CLEAN, columns=["NB_EgeId", "NB_PmSmsseId"])

df_perimetre = construire_perimetre_abc(df_pm, df_ege, ids_valides)

PM totales   : 56,193
PM à traiter : 17,408  (exclues, validées en P1 : 38,785)
   A (avec établissement) : 12,637
   B (sans EG, avec SIREN): 4,154
   C (sans EG, sans SIREN): 617


In [3]:
afficher_tableau(
    df_perimetre[["NB_PmSmsseId", "TX_NumFinessPm", "siren_norm_pm",
                  "TX_DenominationPm", "cdcommune_norm_pm", "dept_pm",
                  "sous_ensemble"]],
    f"Apercu du perimetre ({len(df_perimetre):,} lignes)")

df_perimetre.to_parquet(SN_PERIMETRE, index=False)
print(f"Sauvegarde : {SN_PERIMETRE}")

NB_PmSmsseId,TX_NumFinessPm,siren_norm_pm,TX_DenominationPm,cdcommune_norm_pm,dept_pm,sous_ensemble
6,010000222,546120122,CLINIQUE DU SOUFFLE LE PONTET,01185,01,A
25,010000438,260100243,EHPAD ST TRIVIER DE COURTES,01388,01,A
83,010001022,260104526,MAPA PLEIN SOLEIL LHUIS,01216,01,A
92,010001113,351473251,ASS ATELIER PROTEGE ST RAMBERT,01384,01,A
96,010001154,390533719,RESIDENCE CAMILLE CORNIER,01072,01,A


Sauvegarde : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/sirenisation_perimetre_abc.parquet
